# TP4 — Mobilité et handover LTE (4G)

Analyse des résultats exportés par `tp-export` (scalaires) et `tp-export-vec` (séries temporelles de ue11). Complétez les `# TODO` et les cellules **Réponse**.

In [ ]:
# Environnement : rend visibles les bibliothèques de l'image (pandas, matplotlib) quel que soit le noyau choisi
import sys, glob
sys.path += glob.glob('/home/opp_env/.venv/lib/python3.*/site-packages')
import sys; sys.path.append('/tp/common')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from tpanalyse import load_scalars, load_vectors, kpi_par_run
plt.rcParams['figure.figsize'] = (11, 4)

## 1. Voir le handover (config `AvecHO`)
Après :
```
tp-run AvecHO
tp-export results/AvecHO avecho.csv
tp-export-vec servingCell:vector results/AvecHO cell.csv
tp-export-vec distance:vector results/AvecHO dist.csv
tp-export-vec voIPFrameDelay:vector results/AvecHO delay.csv
```

In [ ]:
cell = load_vectors('cell.csv');  cell = cell[cell.module.str.contains('ue11')]
dist = load_vectors('dist.csv');  dist = dist[dist.module.str.contains('ue11')]
delay = load_vectors('delay.csv'); delay = delay[delay.module.str.contains('ue11')]

fig, ax = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
ax[0].step(cell.t, cell.value, where='post'); ax[0].set_ylabel('Cellule servante'); ax[0].set_yticks([1, 2])
ax[1].plot(dist.t, dist.value); ax[1].set_ylabel('Distance à l\'eNB servante (m)')
ax[2].plot(delay.t, delay.value*1000, '.', ms=2); ax[2].set_ylabel('Délai VoIP (ms)'); ax[2].set_xlabel('Temps (s)'); ax[2].set_yscale('log')
for a in ax: a.grid(alpha=.3)
plt.suptitle('ue11 : handovers entre eNodeB1 et eNodeB2 (10 m/s, couloir 300-700 m)'); plt.tight_layout(); plt.show()

In [ ]:
# Instants et nombre de handovers, position de ue11 à chaque bascule
ho = cell[cell.value.diff().fillna(0) != 0]
print(f'{len(ho)} handovers en {cell.t.max():.0f} s')
print(ho[['t','value']].rename(columns={'value':'nouvelle cellule'}).to_string(index=False))

**Q4.1** — Combien de handovers en 120 s ? À quel instant a lieu le premier, et où se trouve ue11 à ce moment (vitesse × temps) ? Est-ce exactement au milieu (500 m) ? Pourquoi (hystérésis) ?

**Q4.2** — Que se passe-t-il sur la courbe de distance à chaque bascule ? Et sur le délai VoIP ? Estimez la durée de l'interruption à partir des points de délai autour d'un handover.

## 2. Sans handover (config `SansHO`)
Après `tp-run SansHO`, `tp-export results/SansHO sansho.csv` et les mêmes `tp-export-vec` vers `cell_sans.csv`, `dist_sans.csv`, `delay_sans.csv` :

In [ ]:
d2 = load_vectors('delay_sans.csv'); d2 = d2[d2.module.str.contains('ue11')]
ds = load_vectors('dist_sans.csv'); ds = ds[ds.module.str.contains('ue11')]
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].plot(ds.t, ds.value); ax[0].set_ylabel('Distance à eNodeB1 (m)')
ax[1].plot(delay.t, delay.value*1000, '.', ms=2, label='avec handover'); ax[1].plot(d2.t, d2.value*1000, '.', ms=2, label='sans handover')
ax[1].set_ylabel('Délai VoIP (ms)'); ax[1].set_yscale('log'); ax[1].legend(); ax[1].set_xlabel('Temps (s)')
for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.show()

for nom, f in (('avec HO','avecho.csv'), ('sans HO','sansho.csv')):
    sca, iv = load_scalars(f); s = sca[sca.module.str.contains('ue11')]
    print(f"{nom:8s} délai moyen = {1000*s[s.name=='voIPFrameDelay:mean'].value.iloc[0]:.1f} ms   perte playout = {100*s[s.name=='voIPPlayoutLoss:mean'].value.iloc[0]:.1f} %   MOS = {s[s.name=='voIPMos:mean'].value.iloc[0]:.2f}")

**Q4.3** — Sans handover, que devient la voix quand ue11 s'approche d'eNodeB2 ? Pourquoi le délai explose-t-il alors que la cellule n'est pas chargée (CQI, retransmissions HARQ, interférence d'eNodeB2) ? Comparez délai moyen, perte et MOS sur les 120 s.

## 3. Effet de la vitesse (config `Vitesse`, runs 0..3)
Nombre de traversées de x = 500 m en 120 s : départ à 300 m, aller de 400 m ⇒ première à 200/v, puis toutes les 400/v secondes.
Après `tp-run Vitesse 0..3` et `tp-export results/Vitesse vitesse.csv` :

In [ ]:
sca, iv = load_scalars('vitesse.csv')
ue11 = sca[sca.module.str.contains('ue11')]
res = pd.DataFrame({
  'délai (ms)': kpi_par_run(ue11, iv, 'voIPFrameDelay:mean').set_index('speed')['voIPFrameDelay:mean']*1000,
  'perte playout (%)': kpi_par_run(ue11, iv, 'voIPPlayoutLoss:mean').set_index('speed')['voIPPlayoutLoss:mean']*100,
  'MOS': kpi_par_run(ue11, iv, 'voIPMos:mean').set_index('speed')['voIPMos:mean'],
}).sort_index()
res['handovers (120 s)'] = [int((120*v + 200)//400) for v in res.index]   # traversées de x = 500 m : départ 300 m, aller de 400 m
res

**Q4.4** — Le nombre de handovers croît avec la vitesse. La perte et le MOS se dégradent-ils proportionnellement ? À 30 m/s (108 km/h), quelle fraction du temps ue11 passe-t-il en interruption (nombre de HO × latence / 120 s) ?

## 4. Réglages du handover (configs `Latence` et `Mesure`)
Après `tp-run Latence 0..3`, `tp-export results/Latence latence.csv`, `tp-run Mesure 0..2`, `tp-export results/Mesure mesure.csv` :

In [ ]:
for f, var, label in (('latence.csv','hoLatency','Latence de handover (ms)'), ('mesure.csv','bcast','Période des mesures (s)')):
    sca, iv = load_scalars(f); ue11 = sca[sca.module.str.contains('ue11')]
    r = pd.DataFrame({
      'perte playout (%)': kpi_par_run(ue11, iv, 'voIPPlayoutLoss:mean').set_index(var)['voIPPlayoutLoss:mean']*100,
      'MOS': kpi_par_run(ue11, iv, 'voIPMos:mean').set_index(var)['voIPMos:mean'],
    }).sort_index()
    print(label); display(r.round(2))

**Q4.5** — La latence de handover est le temps pendant lequel l'UE n'est rattaché à aucune cellule. Comment la perte évolue-t-elle avec elle ? Quelle latence maximale tolérer pour garder un MOS > 3,5 à 10 m/s ?

**Q4.6** — La période des mesures (`broadcastMessageInterval`) fixe la réactivité de la décision. Avec une mesure toutes les secondes à 10 m/s, de combien de mètres l'UE peut-il dépasser la frontière avant de basculer ? Que se passerait-il à 30 m/s ? Quel est le compromis (signalisation vs réactivité) ?

## 5. Pour aller plus loin (optionnel)
- Rapprochez les eNB (eNodeB2 à 500 m) : ping-pong ? Observez `servingCell` près de la frontière.
- Remettez `shadowing = true` : les handovers deviennent-ils irréguliers ? C'est le problème que résolvent hystérésis et *time-to-trigger* dans les vrais réseaux.

## 6. Synthèse (10 lignes)
- Ce que coûte un handover (interruption, perte) et ce qu'il évite :
- Les deux réglages qui pilotent le compromis réactivité / stabilité :
- Pourquoi la 5G cherche-t-elle à réduire l'interruption (handover conditionnel, DAPS) :